# Regression tests

In [1]:
import numpy as np
import pandas as pd
import torch
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
import matplotlib.pyplot as plt

In [2]:
Bikes_2019 = pd.read_csv("data/CityBikes_2019-06.csv")
malmi = pd.read_csv("data/Malmi_2023-2025.csv")
malmi = malmi[malmi["Vallitseva sää"].isin(['Heikkoa lumisadetta', 'Heikkoja vesikuuroja', 'Utua'])]
diagnoosit = pd.read_csv("data/diagnoosit.csv")
FI_diagnoosit2 = pd.read_csv("data/FI_diagnoosit2.csv")
EN_diagnoosit2 = pd.read_csv("data/EN_diagnoosit2.csv")

## Models

In [3]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
TabPFN_regressor = TabPFNRegressor()

from tabicl import TabICLRegressor
TabICL_regressor = TabICLRegressor()

from sap_rpt_oss import SAP_RPT_OSS_Regressor
ConTextTab_regressor = SAP_RPT_OSS_Regressor(max_context_size=8192, bagging=8)

from catboost import CatBoostRegressor
cb_regressor = CatBoostRegressor()

import xgboost as xgb
xgb_regressor = xgb.XGBRegressor(n_estimators=100)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Evaluate models

In [4]:
def eval_model(y_true, y_pred, model, n):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    print(f"{model} when n={n} R2-score: {r2}, mean absolute error: {mae}, mean squared error: {mse}")

## Train and test models

In [5]:
def test_models(n, column, size, df):
    test_df = df.iloc[:n]
    X = test_df.drop(column, axis=1)
    y = test_df[column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=size, random_state=25)
    TabPFN_reg = TabPFN_regressor.fit(X_train, y_train)
    TabICL_reg = TabICL_regressor.fit(X_train, y_train)
    ConTextTab_reg = ConTextTab_regressor.fit(X_train, y_train)

    TabPFN_pred = TabPFN_reg.predict(X_test)
    TabICL_pred = TabICL_reg.predict(X_test)
    ConTextTab_pred = ConTextTab_reg.predict(X_test)
    
    eval_model(y_test, TabPFN_pred, "TabPFN", n)
    eval_model(y_test, TabICL_pred, "TabICL", n)
    eval_model(y_test, ConTextTab_pred, "ConTextTab", n)

In [6]:
def test_traditional(n, column, size, df):
    test_df = df.iloc[:n]
    X = test_df.drop(column, axis=1)
    y = test_df[column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=size, random_state=25)
    cb_reg = cb_regressor.fit(X_train, y_train, verbose = False)
    xgb_reg = xgb_regressor.fit(X_train, y_train)

    cb_pred = cb_reg.predict(X_test)
    xgb_pred = xgb_reg.predict(X_test)
    
    eval_model(y_test, cb_pred, "CatBoost", n)
    eval_model(y_test, xgb_pred, "XGBoost", n)

In [7]:
def modify_df(df):
    df = df.replace("-", None)
    df["Pilvisyys [1/8]"] = df["Pilvisyys [1/8]"].str[-4].astype(int)
    df_encoded = pd.get_dummies(df["Vallitseva sää"], dtype=int)
    df = df.join(df_encoded)
    df[["Ilman lämpötila keskiarvo [°C]", "Näkyvyys keskiarvo [m]", "Suhteellinen kosteus keskiarvo [%]", "Keskituulen nopeus keskiarvo [m/s]", "Tuulen suunta keskiarvo [°]"]]=df[["Ilman lämpötila keskiarvo [°C]", "Näkyvyys keskiarvo [m]", "Suhteellinen kosteus keskiarvo [%]", "Keskituulen nopeus keskiarvo [m/s]", "Tuulen suunta keskiarvo [°]"]].astype(float)
    df.columns = df.columns.str.replace("[", "(")
    df.columns = df.columns.str.replace("]", ")")
    df = df.drop("Vallitseva sää", axis=1)
    return df

In [8]:
malmi2 = modify_df(malmi)

## Test 1 Bikes

In [9]:
test_models(100, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=100 R2-score: 0.9305719573375244, mean absolute error: 410.6716995239258, mean squared error: 296752.629969325
TabICL when n=100 R2-score: 0.6866429360586018, mean absolute error: 651.4996948242188, mean squared error: 1339365.6119061918
ConTextTab when n=100 R2-score: 0.9075694352284376, mean absolute error: 441.4, mean squared error: 395071.1


## Test 2 Bikes

In [10]:
test_models(1000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=1000 R2-score: 0.6912532013924415, mean absolute error: 590.5793113023043, mean squared error: 1038991.9541279041
TabICL when n=1000 R2-score: 0.7511825496350272, mean absolute error: 536.5021461308003, mean squared error: 837318.2495875022
ConTextTab when n=1000 R2-score: 0.7085544299948834, mean absolute error: 564.855, mean squared error: 980770.015


## Test 3 Bikes

In [11]:
test_models(2000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=2000 R2-score: 0.7027387906460445, mean absolute error: 507.8925124583766, mean squared error: 684335.3010955038
TabICL when n=2000 R2-score: 0.6896091578990691, mean absolute error: 509.6876600241661, mean squared error: 714561.4823005873
ConTextTab when n=2000 R2-score: 0.7060003782835638, mean absolute error: 500.8875, mean squared error: 676826.6875


## Test 4 Bikes

In [12]:
test_models(4000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=4000 R2-score: 0.7346616175821608, mean absolute error: 519.2324017553195, mean squared error: 751870.9262383571
TabICL when n=4000 R2-score: 0.722000783666969, mean absolute error: 509.92982673426866, mean squared error: 787747.0510417972
ConTextTab when n=4000 R2-score: 0.7322351968431852, mean absolute error: 522.1320875, mean squared error: 758746.505986125


## Test 5 Bikes

In [13]:
test_models(6000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=6000 R2-score: 0.5420885720291245, mean absolute error: 541.7538155178369, mean squared error: 1684052.8004676031
TabICL when n=6000 R2-score: 0.597297587455119, mean absolute error: 494.72013857894734, mean squared error: 1481011.5759862643
ConTextTab when n=6000 R2-score: 0.5579732632158187, mean absolute error: 509.39722499999993, mean squared error: 1625633.9512240833


## Test 6 Bikes

In [14]:
test_models(8000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=8000 R2-score: 0.631569591142187, mean absolute error: 482.5484562596411, mean squared error: 1219042.6872162772
TabICL when n=8000 R2-score: 0.6670950138704332, mean absolute error: 440.50489501935544, mean squared error: 1101498.0824661064
ConTextTab when n=8000 R2-score: 0.6273867424787524, mean absolute error: 455.73833125000004, mean squared error: 1232882.6715180625


## Test 7 Bikes

In [15]:
test_models(9000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=9000 R2-score: 0.7239675355451538, mean absolute error: 517.7336922360108, mean squared error: 833278.4467682139
TabICL when n=9000 R2-score: 0.7479529103239593, mean absolute error: 478.8734171773804, mean squared error: 760872.1235471063
ConTextTab when n=9000 R2-score: 0.7541543051706665, mean absolute error: 485.4040777777778, mean squared error: 742151.5405321111


## Test 8 Bikes

In [16]:
test_models(10000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=10000 R2-score: 0.7407744180218564, mean absolute error: 479.9739356983304, mean squared error: 694282.4648936355
TabICL when n=10000 R2-score: 0.7746261022567542, mean absolute error: 440.7417901058197, mean squared error: 603617.6833081999
ConTextTab when n=10000 R2-score: 0.7738435319874051, mean absolute error: 445.798, mean squared error: 605713.637


## Test 9 Bikes

In [17]:
test_models(11000, "Covered distance (m)", 0.2, Bikes_2019)

TabPFN when n=11000 R2-score: 0.7246335099728731, mean absolute error: 485.22504503678954, mean squared error: 842051.5799092774
TabICL when n=11000 R2-score: 0.7527592875667468, mean absolute error: 452.11504013544544, mean squared error: 756044.9076494641
ConTextTab when n=11000 R2-score: 0.7242017587608149, mean absolute error: 458.6368545454546, mean squared error: 843371.8450955227


## Test 1 Weather

In [18]:
test_models(100, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(100, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=100 R2-score: 0.702757716178894, mean absolute error: 0.830344557762146, mean squared error: 1.847241759300232
TabICL when n=100 R2-score: 0.6728459000587463, mean absolute error: 0.8678910136222839, mean squared error: 2.0331318378448486
ConTextTab when n=100 R2-score: 0.8513371348381042, mean absolute error: 0.470161110162735, mean squared error: 0.9238802194595337
CatBoost when n=100 R2-score: 0.8736011137341906, mean absolute error: 0.5198186568391274, mean squared error: 0.7855185185874993
XGBoost when n=100 R2-score: 0.8421801403186986, mean absolute error: 0.4619924134456233, mean squared error: 0.9807872999754155


## Test 2 Weather

In [19]:
test_models(1000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(1000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=1000 R2-score: 0.9543927907943726, mean absolute error: 1.2690653800964355, mean squared error: 3.5722217559814453
TabICL when n=1000 R2-score: 0.9628098011016846, mean absolute error: 1.0882811546325684, mean squared error: 2.9129505157470703
ConTextTab when n=1000 R2-score: 0.982973039150238, mean absolute error: 0.7328681349754333, mean squared error: 1.3336514234542847
CatBoost when n=1000 R2-score: 0.9607237475248638, mean absolute error: 1.160002834886101, mean squared error: 3.076344671823014
XGBoost when n=1000 R2-score: 0.9478046909371447, mean absolute error: 1.2880083951875565, mean squared error: 4.088240369452746


## Test 3 Weather

In [20]:
test_models(2000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(2000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=2000 R2-score: 0.9645015001296997, mean absolute error: 1.1792205572128296, mean squared error: 2.996588706970215
TabICL when n=2000 R2-score: 0.9511845111846924, mean absolute error: 1.4256402254104614, mean squared error: 4.120733261108398
ConTextTab when n=2000 R2-score: 0.9816640019416809, mean absolute error: 0.7669200301170349, mean squared error: 1.5478241443634033
CatBoost when n=2000 R2-score: 0.9681979677929643, mean absolute error: 1.115173038914982, mean squared error: 2.6845523217699836
XGBoost when n=2000 R2-score: 0.9609957517169178, mean absolute error: 1.2495475012203678, mean squared error: 3.2925237168987023


## Test 4 Weather

In [21]:
test_models(4000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(4000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=4000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=4000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=4000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=4000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=4000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 5 Weather

In [22]:
test_models(6000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(6000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=6000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=6000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=6000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=6000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=6000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 6 Weather

In [23]:
test_models(8000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(8000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=8000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=8000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=8000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=8000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=8000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 7 Weather

In [24]:
test_models(9000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(9000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=9000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=9000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=9000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=9000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=9000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 8 Weather

In [25]:
test_models(10000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(10000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=10000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=10000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=10000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=10000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=10000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 9 Weather

In [26]:
test_models(11000, "Ilman lämpötila keskiarvo [°C]", 0.2, malmi)
test_traditional(11000, "Ilman lämpötila keskiarvo (°C)", 0.2, malmi2)

TabPFN when n=11000 R2-score: 0.9589686989784241, mean absolute error: 1.179152250289917, mean squared error: 3.171973943710327
TabICL when n=11000 R2-score: 0.9372302889823914, mean absolute error: 1.4461771249771118, mean squared error: 4.852489948272705
ConTextTab when n=11000 R2-score: 0.9780833721160889, mean absolute error: 0.8574293255805969, mean squared error: 1.6942932605743408
CatBoost when n=11000 R2-score: 0.9512026466263899, mean absolute error: 1.28104296975297, mean squared error: 3.7723387316600507
XGBoost when n=11000 R2-score: 0.9453202529358697, mean absolute error: 1.3707218112232793, mean squared error: 4.227084327875566


## Test 1 diagnoosit

In [27]:
test_models(100, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(100, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=100 R2-score: 0.13426282985998483, mean absolute error: 14.097640289306643, mean squared error: 305.46768485815386
TabICL when n=100 R2-score: 0.026000348258397254, mean absolute error: 15.012442016601563, mean squared error: 343.66714163611215
ConTextTab when n=100 R2-score: -0.08618721166449927, mean absolute error: 16.475625, mean squared error: 383.25152749999995
TabPFN when n=100 R2-score: 0.05797403648202526, mean absolute error: 8.614948299407958, mean squared error: 168.67589525179878
TabICL when n=100 R2-score: -0.6445647351779291, mean absolute error: 11.003745845794677, mean squared error: 294.47004620736396
ConTextTab when n=100 R2-score: -0.18192385670537425, mean absolute error: 9.77296875, mean squared error: 211.63117830078122


## Test 2 diagnoosit

In [28]:
test_models(1000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(1000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=1000 R2-score: 0.2381864379619706, mean absolute error: 13.484066913604735, mean squared error: 358.71659961401804
TabICL when n=1000 R2-score: 0.21267728292018395, mean absolute error: 13.68112943496704, mean squared error: 370.7281439230175
ConTextTab when n=1000 R2-score: 0.21591080877123325, mean absolute error: 14.070343749999997, mean squared error: 369.2055674609375
TabPFN when n=1000 R2-score: 0.6169792286110866, mean absolute error: 5.219990814971924, mean squared error: 87.84630907868257
TabICL when n=1000 R2-score: 0.6098149541673816, mean absolute error: 5.426816264724732, mean squared error: 89.48944468415911
ConTextTab when n=1000 R2-score: 0.6190972163293069, mean absolute error: 5.419296875, mean squared error: 87.36054585742187


## Test 3 diagnoosit

In [29]:
test_models(1600, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(1600, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=1600 R2-score: 0.12957204763812613, mean absolute error: 14.334267805576326, mean squared error: 1022.5532217370104
TabICL when n=1600 R2-score: 0.13870535999239508, mean absolute error: 14.265668043851852, mean squared error: 1011.8236743371983
ConTextTab when n=1600 R2-score: 0.10603983283397167, mean absolute error: 14.420716796875002, mean squared error: 1050.1981772985841
TabPFN when n=1600 R2-score: 0.6271505002417381, mean absolute error: 5.56643884396553, mean squared error: 96.57010680852167
TabICL when n=1600 R2-score: 0.6088269359866725, mean absolute error: 5.71670494055748, mean squared error: 101.31601248459677
ConTextTab when n=1600 R2-score: 0.6268482537099004, mean absolute error: 5.64928515625, mean squared error: 96.6483903515625


## Test 4 diagnoosit

In [30]:
test_models(4000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(4000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=4000 R2-score: 0.18650202723762643, mean absolute error: 13.251580306720735, mean squared error: 603.5196539327691
TabICL when n=4000 R2-score: 0.1877054279118633, mean absolute error: 13.25224624156952, mean squared error: 602.626872410534
ConTextTab when n=4000 R2-score: 0.1567560113972889, mean absolute error: 13.46383203125, mean squared error: 625.5876931743163
TabPFN when n=4000 R2-score: 0.6875507054974723, mean absolute error: 5.221799167442322, mean squared error: 75.20917347722131
TabICL when n=4000 R2-score: 0.6806036499751724, mean absolute error: 5.225417150402069, mean squared error: 76.88138817933616
ConTextTab when n=4000 R2-score: 0.6983310770554746, mean absolute error: 5.1640851562500005, mean squared error: 72.61424735986328


## Test 5 diagnoosit

In [31]:
test_models(6000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(6000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=6000 R2-score: 0.24012422577480086, mean absolute error: 12.807577653948467, mean squared error: 475.846863086309
TabICL when n=6000 R2-score: 0.233587175037263, mean absolute error: 12.893791890970867, mean squared error: 479.9404731115334
ConTextTab when n=6000 R2-score: 0.22771038947641165, mean absolute error: 12.877626041666666, mean squared error: 483.6206140885417
TabPFN when n=6000 R2-score: 0.661419616416991, mean absolute error: 5.15317911605835, mean squared error: 83.49143558771138
TabICL when n=6000 R2-score: 0.6524477815784193, mean absolute error: 5.201455319341024, mean squared error: 85.70382415730653
ConTextTab when n=6000 R2-score: 0.6872505956884964, mean absolute error: 5.123914583333334, mean squared error: 77.12170583789062


## Test 6 diagnoosit

In [32]:
test_models(8000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(8000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=8000 R2-score: 0.30600926259486383, mean absolute error: 12.804814322042466, mean squared error: 318.1256462397453
TabICL when n=8000 R2-score: 0.30314269505166525, mean absolute error: 12.777450922298431, mean squared error: 319.4396820085507
ConTextTab when n=8000 R2-score: 0.31088624559503253, mean absolute error: 12.537962109375, mean squared error: 315.89003517895503
TabPFN when n=8000 R2-score: 0.6832718807633145, mean absolute error: 5.1175277199268345, mean squared error: 77.23392542237454
TabICL when n=8000 R2-score: 0.6844406596584538, mean absolute error: 5.102466569948197, mean squared error: 76.94891952444544
ConTextTab when n=8000 R2-score: 0.6956664739132071, mean absolute error: 5.16980390625, mean squared error: 74.2115127446289


## Test 7 diagnoosit

In [33]:
test_models(9000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(9000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=9000 R2-score: 0.25879449395983267, mean absolute error: 12.774588364749484, mean squared error: 425.79390081816473
TabICL when n=9000 R2-score: 0.2565715008708549, mean absolute error: 12.804416624238755, mean squared error: 427.07092438468476
ConTextTab when n=9000 R2-score: 0.2490099906908636, mean absolute error: 12.808517013888888, mean squared error: 431.41471957964416
TabPFN when n=9000 R2-score: 0.7027105407215227, mean absolute error: 5.00231849454244, mean squared error: 74.03652716535163
TabICL when n=9000 R2-score: 0.7053108899921873, mean absolute error: 4.92203305782742, mean squared error: 73.38894002962134
ConTextTab when n=9000 R2-score: 0.7117217167796728, mean absolute error: 5.068492708333334, mean squared error: 71.79239720985244


## Test 8 diagnoosit

In [34]:
test_models(10000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(10000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=10000 R2-score: 0.2754323245337741, mean absolute error: 12.774647761917114, mean squared error: 411.2144278085421
TabICL when n=10000 R2-score: 0.2784250513359975, mean absolute error: 12.737251365432739, mean squared error: 409.5159633569342
ConTextTab when n=10000 R2-score: 0.2899052922314598, mean absolute error: 12.532044375, mean squared error: 403.000573765625
TabPFN when n=10000 R2-score: 0.669178228073723, mean absolute error: 5.122901607589721, mean squared error: 86.66335198646632
TabICL when n=10000 R2-score: 0.6786713401736136, mean absolute error: 5.053364419784546, mean squared error: 84.17649959289669
ConTextTab when n=10000 R2-score: 0.6732629978571519, mean absolute error: 5.1998146875, mean squared error: 85.59328988183594


## Test 9 diagnoosit

In [35]:
test_models(11000, "ka_maksettu/paiva", 0.2, diagnoosit)
test_models(11000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)

TabPFN when n=11000 R2-score: 0.32441273056843944, mean absolute error: 12.780760062165694, mean squared error: 305.7572991670976
TabICL when n=11000 R2-score: 0.330097158984187, mean absolute error: 12.69760529386347, mean squared error: 303.1846404472699
ConTextTab when n=11000 R2-score: 0.3399618574187153, mean absolute error: 12.676477201704543, mean squared error: 298.7200750433405
TabPFN when n=11000 R2-score: 0.6840376912788044, mean absolute error: 4.988376360043613, mean squared error: 80.55567582566412
TabICL when n=11000 R2-score: 0.6866627460212595, mean absolute error: 4.899178749067133, mean squared error: 79.88640910295373
ConTextTab when n=11000 R2-score: 0.6819047927588526, mean absolute error: 5.047820596590908, mean squared error: 81.09946562906717


## Test 1 diagnoosit (FI vs. EN)

In [45]:
test_models(200, "ka_maksettu/paiva", 0.5, FI_diagnoosit2)
test_models(200, "average_amount_paid/day", 0.5, EN_diagnoosit2)

TabPFN when n=200 R2-score: 0.4112071509359062, mean absolute error: 8.137013979339601, mean squared error: 131.13501174962127
TabICL when n=200 R2-score: 0.1733530855688331, mean absolute error: 9.012752574539185, mean squared error: 184.10949285309505
ConTextTab when n=200 R2-score: 0.3167894684972834, mean absolute error: 8.484275000000002, mean squared error: 152.16356859375
TabPFN when n=200 R2-score: 0.4082008461748079, mean absolute error: 8.130919915771486, mean squared error: 131.80457118940762
TabICL when n=200 R2-score: 0.18293797946020562, mean absolute error: 8.763009916687013, mean squared error: 181.9747604509235
ConTextTab when n=200 R2-score: 0.3102683708301074, mean absolute error: 8.6943, mean squared error: 153.6159371484375


## Test 2 diagnoosit (FI vs. EN)

In [37]:
test_models(1000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(1000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=1000 R2-score: 0.6169792286110866, mean absolute error: 5.219990814971924, mean squared error: 87.84630907868257
TabICL when n=1000 R2-score: 0.6098149541673816, mean absolute error: 5.426816264724732, mean squared error: 89.48944468415911
ConTextTab when n=1000 R2-score: 0.6190972163293069, mean absolute error: 5.419296875, mean squared error: 87.36054585742187
TabPFN when n=1000 R2-score: 0.6150085741688649, mean absolute error: 5.2549326038360595, mean squared error: 88.29828122262384
TabICL when n=1000 R2-score: 0.6134642677462686, mean absolute error: 5.428788919067383, mean squared error: 88.65246989709088
ConTextTab when n=1000 R2-score: 0.6402881235355967, mean absolute error: 5.1801625, mean squared error: 82.50038389453125


## Test 3 diagnoosit (FI vs. EN)

In [38]:
test_models(1600, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(1600, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=1600 R2-score: 0.6271505002417381, mean absolute error: 5.56643884396553, mean squared error: 96.57010680852167
TabICL when n=1600 R2-score: 0.6088269359866725, mean absolute error: 5.71670494055748, mean squared error: 101.31601248459677
ConTextTab when n=1600 R2-score: 0.6268482537099004, mean absolute error: 5.64928515625, mean squared error: 96.6483903515625
TabPFN when n=1600 R2-score: 0.6296135027847183, mean absolute error: 5.524121390342712, mean squared error: 95.93217536755293
TabICL when n=1600 R2-score: 0.6112263047393547, mean absolute error: 5.712805820703506, mean squared error: 100.69456255139376
ConTextTab when n=1600 R2-score: 0.6239552234145596, mean absolute error: 5.644374999999999, mean squared error: 97.39770138671875


## Test 4 diagnoosit (FI vs. EN)

In [39]:
test_models(4000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(4000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=4000 R2-score: 0.6875507054974723, mean absolute error: 5.221799167442322, mean squared error: 75.20917347722131
TabICL when n=4000 R2-score: 0.6806036499751724, mean absolute error: 5.225417150402069, mean squared error: 76.88138817933616
ConTextTab when n=4000 R2-score: 0.6983310770554746, mean absolute error: 5.1640851562500005, mean squared error: 72.61424735986328
TabPFN when n=4000 R2-score: 0.6883359663074704, mean absolute error: 5.2105976804733265, mean squared error: 75.02015459472378
TabICL when n=4000 R2-score: 0.6866391847871995, mean absolute error: 5.186341191673279, mean squared error: 75.42858417979991
ConTextTab when n=4000 R2-score: 0.6899345350217598, mean absolute error: 5.29164375, mean squared error: 74.63536565820314


## Test 5 diagnoosit (FI vs. EN)

In [40]:
test_models(6000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(6000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=6000 R2-score: 0.661419616416991, mean absolute error: 5.15317911605835, mean squared error: 83.49143558771138
TabICL when n=6000 R2-score: 0.6524477815784193, mean absolute error: 5.201455319341024, mean squared error: 85.70382415730653
ConTextTab when n=6000 R2-score: 0.6872505956884964, mean absolute error: 5.123914583333334, mean squared error: 77.12170583789062
TabPFN when n=6000 R2-score: 0.662354590818846, mean absolute error: 5.12601510105133, mean squared error: 83.26087776796247
TabICL when n=6000 R2-score: 0.6543487863925836, mean absolute error: 5.177027749506633, mean squared error: 85.23505033374911
ConTextTab when n=6000 R2-score: 0.6764294633417646, mean absolute error: 5.1665197916666665, mean squared error: 79.7901175891927


## Test 6 diagnoosit (FI vs. EN)

In [41]:
test_models(8000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(8000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=8000 R2-score: 0.6832718807633145, mean absolute error: 5.1175277199268345, mean squared error: 77.23392542237454
TabICL when n=8000 R2-score: 0.6844406596584538, mean absolute error: 5.102466569948197, mean squared error: 76.94891952444544
ConTextTab when n=8000 R2-score: 0.6956664739132071, mean absolute error: 5.16980390625, mean squared error: 74.2115127446289
TabPFN when n=8000 R2-score: 0.6834234223073862, mean absolute error: 5.108484995365143, mean squared error: 77.19697212520144
TabICL when n=8000 R2-score: 0.6891543568832204, mean absolute error: 5.057689111948013, mean squared error: 75.79948782637379
ConTextTab when n=8000 R2-score: 0.6873116556700842, mean absolute error: 5.180150781249999, mean squared error: 76.24882919970703


## Test 7 diagnoosit (FI vs. EN)

In [42]:
test_models(9000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(9000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=9000 R2-score: 0.7027105407215227, mean absolute error: 5.00231849454244, mean squared error: 74.03652716535163
TabICL when n=9000 R2-score: 0.7053108899921873, mean absolute error: 4.92203305782742, mean squared error: 73.38894002962134
ConTextTab when n=9000 R2-score: 0.7117217167796728, mean absolute error: 5.068492708333334, mean squared error: 71.79239720985244
TabPFN when n=9000 R2-score: 0.7051871273379251, mean absolute error: 4.9790755045996775, mean squared error: 73.41976169795959
TabICL when n=9000 R2-score: 0.7078429164454636, mean absolute error: 4.892678956264921, mean squared error: 72.75836790726507
ConTextTab when n=9000 R2-score: 0.7112536502064805, mean absolute error: 5.033615972222223, mean squared error: 71.90896381684027


## Test 8 diagnoosit (FI vs. EN)

In [43]:
test_models(10000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(10000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=10000 R2-score: 0.669178228073723, mean absolute error: 5.122901607589721, mean squared error: 86.66335198646632
TabICL when n=10000 R2-score: 0.6786713401736136, mean absolute error: 5.053364419784546, mean squared error: 84.17649959289669
ConTextTab when n=10000 R2-score: 0.6732629978571519, mean absolute error: 5.1998146875, mean squared error: 85.59328988183594
TabPFN when n=10000 R2-score: 0.67034538248492, mean absolute error: 5.094328090896607, mean squared error: 86.35759969884884
TabICL when n=10000 R2-score: 0.6828419307740831, mean absolute error: 5.02455142906189, mean squared error: 83.08395553482161
ConTextTab when n=10000 R2-score: 0.6816731172681729, mean absolute error: 5.1384637500000006, mean squared error: 83.39014244531249


## Test 9 diagnoosit (FI vs. EN)

In [44]:
test_models(11000, "ka_maksettu/paiva", 0.2, FI_diagnoosit2)
test_models(11000, "average_amount_paid/day", 0.2, EN_diagnoosit2)

TabPFN when n=11000 R2-score: 0.6840376912788044, mean absolute error: 4.988376360043613, mean squared error: 80.55567582566412
TabICL when n=11000 R2-score: 0.6866627460212595, mean absolute error: 4.899178749067133, mean squared error: 79.88640910295373
ConTextTab when n=11000 R2-score: 0.6819047927588526, mean absolute error: 5.047820596590908, mean squared error: 81.09946562906717
TabPFN when n=11000 R2-score: 0.6841293188108861, mean absolute error: 4.9739546538959845, mean squared error: 80.53231507165218
TabICL when n=11000 R2-score: 0.6899216952357756, mean absolute error: 4.8651374208450315, mean squared error: 79.05552880739128
ConTextTab when n=11000 R2-score: 0.6847717437054521, mean absolute error: 5.016700852272727, mean squared error: 80.36852663828348
